<a href="https://colab.research.google.com/github/KyberNull/embeddinggemma2_asr/blob/main/DSP_Mini_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EmbeddingGemma 2 based ASR (TPU Baseline — Streaming)

This notebook is a **TPU-oriented ML baseline** for the Speech-to-Text stage of the project:

- **Dataset:** Streams real LibriSpeech (`openslr/librispeech_asr`, `clean` config) using speaker-disjoint `train.100` and `validation` splits—no full 30 GB download required.
- **Audio Encoder:** Frozen `google/embeddinggemma-2` Conformer `audio_tower` (16 kHz mono input, ~25 fps), keeping only the ~300M audio tower on TPU.
- **Layer Mixing:** Extracts temporally ordered intermediate Conformer hidden states (cached as `bfloat16`) and learns a softmax layer mixture.
- **Reader / Adapter:** LayerNorm + projection + causal depthwise Conv1D + masked Bidirectional GRU.
- **Output Head:** Grapheme-level CTC output (Unicode `\X` clusters for English and Indic scripts such as Telugu).

## Colab setup

1. Open **Runtime → Change runtime type → Hardware accelerator → TPU**.
2. Click **Runtime → Run all**.
3. Adjust `MAX_TRAIN_UTTERANCES` / `MAX_VAL_UTTERANCES` in Section 3 once the initial streaming smoke test passes.

## 1. Install dependencies with `uv`

Requires `transformers>=5.19.0` for `google/embeddinggemma-2` support and `torch_tpu` for the PyTorch-native TPU backend.

In [1]:
!command -v uv >/dev/null 2>&1 || pip -q install uv
!uv pip install --system -q torch_tpu polars datasets
!uv pip install --system -q -U "transformers>=5.19.0" librosa soundfile tqdm jiwer regex

## 2. Verify TPU runtime

In [2]:
import torch, sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("TPU available:", torch.accelerator.is_available())
print("TPU device count:", torch.accelerator.device_count())

if not torch.accelerator.is_available():
    raise RuntimeError(
        "No TPU detected. In Colab choose Runtime > Change runtime type > TPU, "
        "then restart and rerun the notebook."
    )

DEVICE = torch.device("tpu")
x = torch.randn(128, 128, device=DEVICE)
y = x @ x
torch.accelerator.synchronize()
print("TPU smoke test:", y.device, tuple(y.shape))

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch: 2.14.1+cpu
TPU available: True
TPU device count: 1
TPU smoke test: tpu:0 (128, 128)


## 3. Configure streaming LibriSpeech (`openslr/librispeech_asr`)

Uses Hugging Face `streaming=True` on the official `openslr/librispeech_asr` (`clean`) dataset. `train.100` and `validation` are already speaker-disjoint, so no random split is needed.

In [3]:
from datasets import load_dataset
import os

DATASET_ID = "openslr/librispeech_asr"
DATASET_CONFIG = "clean"

MAX_TRAIN_UTTERANCES = 200  # increase (or set None) after smoke test
MAX_VAL_UTTERANCES = 50     # increase (or set None) after smoke test

CACHE_DIR = "/content/eg2_asr_cache"
os.makedirs(os.path.join(CACHE_DIR, "train"), exist_ok=True)
os.makedirs(os.path.join(CACHE_DIR, "val"), exist_ok=True)

train_stream = load_dataset(DATASET_ID, DATASET_CONFIG, split="train.100", streaming=True)
val_stream = load_dataset(DATASET_ID, DATASET_CONFIG, split="validation", streaming=True)

sample_peek = next(iter(train_stream))
print("Stream connected:", DATASET_ID, f"({DATASET_CONFIG})")
print("Sample keys:", list(sample_peek.keys()))
print("Sample text:", sample_peek["text"])
print("Sample SR:", sample_peek["audio"]["sampling_rate"])

README.md:   0%|          | 0.00/11.0k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Stream connected: openslr/librispeech_asr (clean)
Sample keys: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id']
Sample text: CHAPTER SIXTEEN I MIGHT HAVE TOLD YOU OF THE BEGINNING OF THIS LIAISON IN A FEW LINES BUT I WANTED YOU TO SEE EVERY STEP BY WHICH WE CAME I TO AGREE TO WHATEVER MARGUERITE WISHED
Sample SR: 16000


## 4. Load the EmbeddingGemma 2 audio tower

In `google/embeddinggemma-2`, only `vision_config=None` and `audio_config=None` are valid selective-loading overrides (`text_config=None` is not supported). We load with `vision_config=None` in `bfloat16` (the model card warns `float16` can overflow), keep only `audio_tower` on the TPU, and free the text backbone immediately.

In [4]:
import gc
import torch
from transformers import AutoProcessor, AutoModel

MODEL_ID = "google/embeddinggemma-2"

processor = AutoProcessor.from_pretrained(MODEL_ID)

full_model = AutoModel.from_pretrained(
    MODEL_ID,
    vision_config=None,
    dtype=torch.bfloat16,
)

audio_tower = full_model.audio_tower.to(DEVICE)
audio_tower.eval()
for p in audio_tower.parameters():
    p.requires_grad_(False)

del full_model
gc.collect()

n_params = sum(p.numel() for p in audio_tower.parameters())
print("Loaded audio tower:", type(audio_tower).__name__)
print(f"Parameters: {n_params / 1e6:.1f}M | Device: {next(audio_tower.parameters()).device}")

processor_config.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/511 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.49GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1165 [00:00<?, ?it/s]

Loaded audio tower: Gemma4AudioModel
Parameters: 304.8M | Device: tpu:0


## 5. Stream audio and cache intermediate Conformer hidden states

Streams utterances one by one from Hugging Face without buffering raw audio arrays in RAM. Each utterance's intermediate layer stack `[layers, time_steps, hidden_dim]` is saved as `bfloat16` on local Colab disk, and lightweight metadata is stored in Polars DataFrames.

In [7]:
import inspect
import itertools
import librosa
import numpy as np
import polars as pl
import torch
from tqdm.auto import tqdm

SAMPLE_RATE = 16000
_TOWER_SIG = inspect.signature(audio_tower.forward).parameters

def prepare_audio(audio_dict):
    wav = np.asarray(audio_dict["array"], dtype=np.float32)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    sr = audio_dict["sampling_rate"]
    if sr != SAMPLE_RATE:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    return np.asarray(wav, dtype=np.float32)

@torch.inference_mode()
def extract_hidden_states(wav):
    batch = processor(
        audio=[wav],
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
    )
    input_features = batch["input_features"].to(DEVICE, dtype=torch.bfloat16)
    mask = batch.get("input_features_mask")
    if mask is None:
        mask = batch.get("attention_mask")
    if mask is not None:
        mask = mask.to(DEVICE)

    kwargs = {"output_hidden_states": True, "return_dict": True}
    if "input_features_mask" in _TOWER_SIG and mask is not None:
        kwargs["input_features_mask"] = mask
    elif "attention_mask" in _TOWER_SIG and mask is not None:
        kwargs["attention_mask"] = mask

    out = audio_tower(input_features, **kwargs)

    if isinstance(out, tuple):
        hs = (out[0],)
    else:
        hs = getattr(out, "hidden_states", None)
        if hs is None:
            hs = (out.last_hidden_state,)

    # Keep only the Conformer layers with matching hidden dim (drops the 1536-D final projection)
    base_dim = hs[0].shape[-1]
    hs = [h[0] for h in hs if h.shape[-1] == base_dim]

    stacked = torch.stack(hs, dim=0)  # [layers, T, 1024]
    return stacked.detach().to("cpu", dtype=torch.bfloat16).contiguous()

def cache_split(stream, split_name, max_utterances):
    split_dir = os.path.join(CACHE_DIR, split_name)
    iterator = stream if max_utterances is None else itertools.islice(stream, max_utterances)
    records, failed = [], []

    for idx, item in enumerate(tqdm(iterator, total=max_utterances, desc=f"Caching {split_name}")):
        text = str(item["text"]).strip().upper()
        if not text:
            continue
        utt_id = str(item.get("id", f"{split_name}_{idx:08d}"))
        cache_path = os.path.join(split_dir, f"utt_{idx:08d}.pt")
        try:
            if not os.path.exists(cache_path):
                wav = prepare_audio(item["audio"])
                feat = extract_hidden_states(wav)
                torch.save(feat, cache_path)
            else:
                feat = torch.load(cache_path, map_location="cpu", weights_only=True)

            records.append({
                "cache_path": cache_path,
                "utt_id": utt_id,
                "speaker_id": int(item.get("speaker_id", -1)),
                "text": text,
                "frames": int(feat.shape[1]),
                "layers": int(feat.shape[0]),
                "dim": int(feat.shape[2]),
            })
        except Exception as e:
            failed.append((utt_id, repr(e)))

    frame = pl.DataFrame(records)
    print(f"[{split_name}] Cached: {len(frame)} | Failed: {len(failed)}")
    if len(frame) > 0:
        sample_shape = torch.load(frame["cache_path"][0], map_location="cpu", weights_only=True).shape
        print(f"[{split_name}] Feature shape [layers, T, D]: {tuple(sample_shape)}")
    if failed[:3]:
        print(f"[{split_name}] First failures:", failed[:3])
    return frame

train_df = cache_split(train_stream, "train", MAX_TRAIN_UTTERANCES)
val_df = cache_split(val_stream, "val", MAX_VAL_UTTERANCES)

Caching train:   0%|          | 0/200 [00:00<?, ?it/s]

[train] Cached: 200 | Failed: 0
[train] Feature shape [layers, T, D]: (12, 364, 1024)


Caching val:   0%|          | 0/50 [00:00<?, ?it/s]

[val] Cached: 50 | Failed: 0
[val] Feature shape [layers, T, D]: (12, 165, 1024)


## 6. Build grapheme-level vocabulary

Uses Unicode grapheme clusters (`\X`) after NFC normalization so the same pipeline works cleanly on both English and Indic scripts (Telugu). ID `0` is reserved for CTC `<blank>`.

In [8]:
import regex, unicodedata, json
from collections import Counter

def normalize_text(s):
    return unicodedata.normalize("NFC", str(s).strip())

def graphemes(s):
    return regex.findall(r"\X", normalize_text(s))

train_texts = [normalize_text(s) for s in train_df["text"].to_list()]
counter = Counter(g for s in train_texts for g in graphemes(s))
itos = ["<blank>"] + sorted(counter.keys())
stoi = {tok: i for i, tok in enumerate(itos)}

print("Grapheme vocabulary size (including blank):", len(itos))
print("Example:", train_texts[0], "->", graphemes(train_texts[0])[:20])

with open(os.path.join(CACHE_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(itos, f, ensure_ascii=False, indent=2)

Grapheme vocabulary size (including blank): 29
Example: CHAPTER SIXTEEN I MIGHT HAVE TOLD YOU OF THE BEGINNING OF THIS LIAISON IN A FEW LINES BUT I WANTED YOU TO SEE EVERY STEP BY WHICH WE CAME I TO AGREE TO WHATEVER MARGUERITE WISHED -> ['C', 'H', 'A', 'P', 'T', 'E', 'R', ' ', 'S', 'I', 'X', 'T', 'E', 'E', 'N', ' ', 'I', ' ', 'M', 'I']


## 7. Dataset and fixed-bucket collate function

- Filters utterances that violate the CTC length constraint (`frames >= len(ids) + adjacent_repeats`) or contain out-of-vocabulary graphemes.
- Pads batch time lengths up to multiples of `BUCKET_STEP = 64` to minimize TPU graph recompilations.

In [9]:
import math
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import numpy as np

BUCKET_STEP = 64  # keeps TPU shapes static across batches

class CachedASRDataset(Dataset):
    def __init__(self, frame):
        self.items = []
        self.rejected = []
        for r in frame.to_dicts():
            toks = graphemes(r["text"])
            if any(t not in stoi for t in toks):
                self.rejected.append((r["utt_id"], "oov_grapheme"))
                continue
            ids = [stoi[t] for t in toks]
            repeats = sum(ids[i] == ids[i - 1] for i in range(1, len(ids)))
            min_frames = len(ids) + repeats
            if r["frames"] < min_frames:
                self.rejected.append((r["utt_id"], f"frames={r['frames']} < min={min_frames}"))
                continue
            self.items.append((r["cache_path"], torch.tensor(ids, dtype=torch.long), r["text"]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        p, y, text = self.items[i]
        return torch.load(p, map_location="cpu", weights_only=True).float(), y, text

def collate_fn(batch):
    feats, targets, texts = zip(*batch)
    B, L, D = len(feats), feats[0].shape[0], feats[0].shape[-1]
    raw_maxT = max(x.shape[1] for x in feats)
    maxT = int(math.ceil(raw_maxT / BUCKET_STEP) * BUCKET_STEP)

    x = torch.zeros(B, L, maxT, D, dtype=torch.float32)
    input_lengths = torch.tensor([z.shape[1] for z in feats], dtype=torch.long)
    for i, z in enumerate(feats):
        x[i, :, :z.shape[1], :] = z.float()

    target_lengths = torch.tensor([len(y) for y in targets], dtype=torch.long)
    flat_targets = torch.cat(targets)
    return x, input_lengths, flat_targets, target_lengths, texts

train_ds = CachedASRDataset(train_df)
val_ds = CachedASRDataset(val_df)
print(f"Train usable: {len(train_ds)} (rejected: {len(train_ds.rejected)})")
print(f"Val usable:   {len(val_ds)} (rejected: {len(val_ds.rejected)})")

BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

Train usable: 200 (rejected: 0)
Val usable:   50 (rejected: 0)


## 8. TPU reader: learned layer mix + projection + causal depthwise Conv1D + masked BiGRU + CTC

In [10]:
import torch.nn as nn

class TPUCTCReader(nn.Module):
    def __init__(self, n_layers, input_dim, vocab_size, hidden=256, gru_layers=2, kernel_size=5):
        super().__init__()
        self.layer_logits = nn.Parameter(torch.zeros(n_layers))
        self.norm_in = nn.LayerNorm(input_dim)
        self.proj = nn.Linear(input_dim, hidden)

        self.dwconv = nn.Conv1d(
            hidden, hidden, kernel_size=kernel_size,
            groups=hidden, padding=0,
        )
        self.conv_act = nn.GELU()

        self.gru = nn.GRU(
            input_size=hidden,
            hidden_size=hidden // 2,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.1 if gru_layers > 1 else 0.0,
        )
        self.norm_out = nn.LayerNorm(hidden)
        self.classifier = nn.Linear(hidden, vocab_size)
        self.kernel_size = kernel_size

    def forward(self, x, input_lengths=None):
        # x: [B, layers, T, D]
        weights = torch.softmax(self.layer_logits, dim=0)
        x = torch.einsum("l,bltd->btd", weights, x)

        mask = None
        if input_lengths is not None:
            T = x.shape[1]
            mask = (
                torch.arange(T, device=x.device)[None, :] < input_lengths.to(x.device)[:, None]
            ).unsqueeze(-1)

        x = self.proj(self.norm_in(x))
        if mask is not None:
            x = x * mask

        x = x.transpose(1, 2)
        x = F.pad(x, (self.kernel_size - 1, 0))  # causal left padding
        x = self.dwconv(x)
        x = self.conv_act(x).transpose(1, 2)
        if mask is not None:
            x = x * mask

        x, _ = self.gru(x)
        if mask is not None:
            x = x * mask

        x = self.norm_out(x)
        logits = self.classifier(x)
        return logits.log_softmax(dim=-1)

sample_x, sample_il, sample_y, sample_tl, _ = next(iter(train_loader))
n_layers, _, input_dim = sample_x.shape[1], sample_x.shape[2], sample_x.shape[3]

reader = TPUCTCReader(
    n_layers=n_layers,
    input_dim=input_dim,
    vocab_size=len(itos),
).to(DEVICE)

with torch.no_grad():
    test_out = reader(sample_x.to(DEVICE), sample_il)
print("Reader output:", test_out.shape, test_out.device)

Reader output: torch.Size([8, 448, 29]) tpu:0


## 9. Train with CTC loss and log validation CER

In [11]:
import torch.optim as optim
from jiwer import cer
from tqdm.auto import tqdm

BLANK_ID = 0
CKPT_PATH = os.path.join(CACHE_DIR, "eg2_ctc_reader.pt")
ctc_loss_fn = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)
optimizer = optim.AdamW(reader.parameters(), lr=3e-4, weight_decay=1e-4)

def ctc_greedy_decode(frame_ids):
    out = []
    prev = None
    for idx in frame_ids:
        idx = int(idx)
        if idx != prev and idx != BLANK_ID:
            out.append(itos[idx])
        prev = idx
    return "".join(out)

def train_one_epoch():
    reader.train()
    losses = []
    for x, input_lengths, targets, target_lengths, _ in tqdm(train_loader, leave=False):
        x = x.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        log_probs = reader(x, input_lengths)
        log_probs_tbc = log_probs.transpose(0, 1)

        loss = ctc_loss_fn(
            log_probs_tbc.float().cpu(),
            targets,
            input_lengths,
            target_lengths,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reader.parameters(), 5.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

@torch.no_grad()
def validate():
    reader.eval()
    losses, refs, hyps = [], [], []
    for x, input_lengths, targets, target_lengths, texts in val_loader:
        x = x.to(DEVICE)
        lp = reader(x, input_lengths)
        loss = ctc_loss_fn(lp.transpose(0, 1).float().cpu(), targets, input_lengths, target_lengths)
        losses.append(float(loss.cpu()))

        preds = lp.cpu().argmax(dim=-1)
        for i in range(preds.shape[0]):
            refs.append(texts[i])
            hyps.append(ctc_greedy_decode(preds[i, :input_lengths[i]].tolist()))

    return float(np.mean(losses)), float(cer(refs, hyps))

EPOCHS = 10
history = []
for epoch in range(EPOCHS):
    tr = train_one_epoch()
    va, va_cer = validate()
    history.append({"epoch": epoch + 1, "train_ctc": tr, "val_ctc": va, "val_cer": va_cer})
    print(history[-1])

    ckpt = {
        "model": {k: v.detach().cpu() for k, v in reader.state_dict().items()},
        "itos": itos,
        "input_dim": input_dim,
        "n_layers": n_layers,
        "epoch": epoch + 1,
        "history": history,
    }
    torch.save(ckpt, CKPT_PATH)

pl.DataFrame(history)

  0%|          | 0/25 [00:00<?, ?it/s]

/tmp/ipykernel_799/3072687198.py:31: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at external/rules_python++pip+torch_tpu_pypi_2_14_0_314_torch/site-packages/torch/include/ATen/core/TensorBody.h:494.)
  log_probs_tbc.float().cpu(),


{'epoch': 1, 'train_ctc': 3.0008530521392824, 'val_ctc': 2.870255878993443, 'val_cer': 1.0}


  0%|          | 0/25 [00:00<?, ?it/s]

{'epoch': 2, 'train_ctc': 2.8282246494293215, 'val_ctc': 2.8039119243621826, 'val_cer': 1.0}


  0%|          | 0/25 [00:00<?, ?it/s]

... (12 outputs left out)

{'epoch': 9, 'train_ctc': 0.8951861333847045, 'val_ctc': 1.7681280204227992, 'val_cer': 0.3037627709942686}


  0%|          | 0/25 [00:00<?, ?it/s]

{'epoch': 10, 'train_ctc': 0.8340107870101928, 'val_ctc': 1.7669026000159127, 'val_cer': 0.29529030650386245}


epoch,train_ctc,val_ctc,val_cer
i64,f64,f64,f64
1,3.000853,2.870256,1.0
2,2.828225,2.803912,1.0
3,2.603254,2.40801,0.78146
4,1.909974,2.010573,0.548218
5,1.419185,1.844385,0.40618
6,1.179659,1.810474,0.355844
7,1.048775,1.773886,0.331174
8,0.959068,1.779625,0.31697
9,0.895186,1.768128,0.303763


## 10. Inspect validation predictions & learned layer weights

In [12]:
@torch.no_grad()
def inspect_predictions(n=10):
    reader.eval()
    shown = 0
    for x, input_lengths, targets, target_lengths, texts in val_loader:
        lp = reader(x.to(DEVICE), input_lengths).cpu()
        pred = lp.argmax(dim=-1)
        for i in range(pred.shape[0]):
            hyp = ctc_greedy_decode(pred[i, :input_lengths[i]].tolist())
            print("REF:", texts[i])
            print("HYP:", hyp)
            print("-" * 60)
            shown += 1
            if shown >= n:
                return

weights = torch.softmax(reader.layer_logits.detach().cpu(), dim=0).tolist()
print("Learned layer mix weights:", [round(w, 4) for w in weights])
print("=" * 60)
inspect_predictions()

Learned layer mix weights: [0.0811, 0.0795, 0.0797, 0.0784, 0.0775, 0.081, 0.0855, 0.0813, 0.0807, 0.0902, 0.092, 0.093]
REF: HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE
HYP: HE WAN A FEVERD STA OF MIDED OWING TO THEBLIGT HS WIE ACTIN THRAN ICAS APON HIS INTIER FUTUER
------------------------------------------------------------
REF: HE WOULD HAVE TO PAY HER THE MONEY WHICH SHE WOULD NOW REGULARLY DEMAND OR THERE WOULD BE TROUBLE IT DID NOT MATTER WHAT HE DID
HYP: WHE WHAETO PAY HE THE MONY WIH H WAL NO RGULARLY OMAND AOR THERWIL E TRUL ID DID NOT MAER WAT HE DID
------------------------------------------------------------
REF: HURSTWOOD WALKED THE FLOOR MENTALLY ARRANGING THE CHIEF POINTS OF HIS SITUATION
HYP: HERST WUD WAE HE LOR MNTLY ARANGIN TH HEPOINCE OF ISHICOU ATION
------------------------------------------------------------
REF: HE ALSO THOUGHT OF HIS MANAGERIAL POSITION
HYP:  HE AL THU OF HIS ANERYALPOSITON